# R07 · Cookbook Reference Lab: Examples, limitations and API reference

**NHyDB 0.27.1 notebook series · v1 · reference part**

These examples are based on the official
[NHyDB Python Cookbook](https://github.com/nhydb/NHyDB/blob/main/docs/cookbook/PYTHON_COOKBOOK.md) (sections 25–27), which serves as their authoritative reference.
Code and expected output are taken unchanged from the cookbook, and the same examples are run by the
NHyDB test suite.

* Every code cell is an **independent example** – run any cell on its own, or *Run All*.
* Each example creates its own temporary database folder.
* Under each example you find the output printed in the cookbook, to compare with yours.
* The main series (`../main/`) explains the ideas behind these APIs step by step.

**Contents** (reference text, no runnable examples)

* 25. Complete application examples
* 26. Known limitations
* 27. API reference

In [1]:
import nhydb

print('nhydb', nhydb.ENGINE_VERSION, '· file format', nhydb.FILE_FORMAT_VERSION)
if nhydb.ENGINE_VERSION != '0.27.1':
    print('⚠ These examples were written for NHyDB 0.27.1; results may differ.')

nhydb 0.27.1 · file format 0.16


## 25. Complete application examples

Runnable programs in [`examples/python/`](https://github.com/nhydb/NHyDB/blob/main/examples/python); each
creates its database in a temporary directory:

| Program | Shows |
|---|---|
| [`sensor_timeseries.py`](https://github.com/nhydb/NHyDB/blob/main/examples/python/sensor_timeseries.py) | NumPy batch writes of a day of sensor data, time-window reads with `read_numpy`, NHySQL aggregation (needs NumPy) |
| [`document_store.py`](https://github.com/nhydb/NHyDB/blob/main/examples/python/document_store.py) | records with extensions, VALUE and FULLTEXT indexes, BM25 search, phrase queries, NHySQL grouping |
| [`math_map_grid.py`](https://github.com/nhydb/NHyDB/blob/main/examples/python/math_map_grid.py) | MATH_MAP planning with `dry_run`, writing and reading cells by input tuples, lookup through the QueryEngine |

## 26. Known limitations

* **One open object per database.** The exclusive lock is taken for every
  open, also for reading; there is no shared read mode and no multi-process
  transaction.
* **No rollback API.** Atomicity comes from the commit boundary;
  multi-block calls are not atomic as a whole (exception: `put_math_batch`).
* **Context manager commits** on exit, even after an exception.
* **Key file.** Without the matching `*.layout.key` the database cannot be
  opened; a lost key cannot be recovered. Only the schema is encrypted, not
  the data.
* **Renamed database files** do not open (use `compact_file(output=...)`).
* **Indexes** are not maintained automatically (`build_indexes`).
* **NHySQL** is read-only and works on CLASSIC layouts; no `DISTINCT`,
  `HAVING` or joins.
* **QueryEngine `order_by`** only takes effect for fields listed in
  `select` (0.27.1).
* **Closed objects:** do not read through a closed database object; a
  previously read record may still be returned from the cache (0.27.1).
* **CUSTOM layouts** need their plugin registered after every open;
  plugins must not write to the same database during a callback.
* **`[cache]` configuration** keys and the `strict` open option are
  accepted but not used in 0.27.1.
* **NumPy coordinate arrays** (`u`, `v`, object ids) are limited to
  `uint64`; use Python `int` lists for larger values.
* **Durability `full`** raises an error on macOS file systems without
  `F_FULLFSYNC` support.

## 27. API reference

Section numbers refer to the recipes above. Parameters in *italics* are
keyword-only. All write methods accept *`commit=False`*; reading and
writing methods with a lane accept *`lane=None`* (`NL`).

### 27.1 Module `nhydb`

| Name | Purpose | Key parameters | See |
|---|---|---|---|
| `create(path, **options)` / `NHyDB.create` | create a database | *overwrite, durability, lock, lock_timeout_ms, layout_path, key_path, config* | 3, 4 |
| `open(path, **options)` / `NHyDB(path, ...)` | open a database | as `create` plus *strict* | 3, 4 |
| `dynadron_n(realm, u, v, lane)` | address → N | | 9 |
| `n_to_dynadron_address(n)` | N (int or `"N…"`) → address | | 9 |
| `parse_n_reference(ref)` | `"N589"` → 589 | | 9 |
| `cell_address(realm, u, v, lane)` | address with all indicators | | 9 |
| `dynadron_address(realm, u, v)` | indicators without lane | | 9 |
| `global_v`, `split_global_v`, `realm_v_bounds`, `validate_local_v` | realm ↔ global `v` | | 9 |
| `normalize_lane`, `lane_name` | lane conversion | | 9 |
| `enc_vu`, `dec_vu`, `enc_vi`, `dec_vi` | VarUInt/VarInt codecs | | 9 |
| `ENGINE_VERSION`, `FILE_FORMAT_VERSION`, `REALM_SPAN`, `TYPE_CODES` | constants | | 1, 9, 11 |
| `LayoutDef`, `IndexDef`, `MathLayoutState` | definition objects returned by the API (`to_dict()`) | | 13, 17, 18 |
| `QueryEngine` | query engine (also `nhydb.query.QueryEngine`) | | 14 |
| exceptions | see the hierarchy | | 22 |

### 27.2 `NHyDB` – life cycle, attributes and maintenance

| Method / attribute | Purpose | Key parameters | See |
|---|---|---|---|
| `commit()` | make data durable; returns the commit boundary | | 5 |
| `sync()` | write buffers (no commit) | *hard=False* | 6 |
| `close()` | release file and lock | *commit=True* | 5 |
| `checkpoint()` | write a checkpoint block and commit | | 20 |
| `closed`, `path`, `layout_path`, `key_path`, `config_source` | state and file locations | | 4 |
| `durability` | `"fsync"`/`"full"`, writable | | 4, 5 |
| `max_db_size_bytes`, `set_max_db_size(size)` | size limit | | 19 |
| `committed_end`, `last_valid_end`, `checkpoint_offset`, `next_block_id`, `recovered_tail_bytes` | container state | | 4, 5, 6, 20 |
| `stats()`, `health()`, `verify()` | statistics and checks | *deep=False* | 20 |
| `NHyDB.inspect_tail(path)` | check the file tail without opening | | 20 |
| `NHyDB.repair_file(path)` | cut off a damaged tail | *apply=False, make_backup=True, rebuild_indexes=True*, side-file options | 20 |
| `NHyDB.compact_file(path)` | rewrite in place or export | *output=None, make_backup=True, rebuild_indexes=True, batch_records=10000*, side-file options | 20, 21 |
| `get_system_metadata()`, `get_system_cell(v)` | system space R0,u1 | | 23 |

### 27.3 `NHyDB` – schema and layouts

| Method / attribute | Purpose | Key parameters | See |
|---|---|---|---|
| `create_realm(realm_id, name)` | define a realm | *description, default_shape_id, base_reserve_to* | 3, 8 |
| `realms`, `realm_bounds(realm_id)` | realm definitions, `v` range | | 4, 9 |
| `create_shape(shape_id, realm_id, name, fields)` | BASE fields `[{v, name, type, enum?}]` | *description* | 3, 6 |
| `set_realm_default_shape(realm_id, shape_id)` | default shape | | 6 |
| `define_extension(realm_id, v, type)`, `define_extensions(realm_id, fields)` | extension fields | *name, enum, description, replace* | 8 |
| `extension_catalog(realm_id)`, `resolve_v(realm_id, column)` | list extensions, name → `v` | | 8 |
| `create_classic_layout(layout_id, realm_id, name, *, shape_id)` | CLASSIC layout | *base_reserve_to, u_start, u_end, priority* | 3, 6 |
| `create_column_layout(..., *, fields)` | COLUMN layout `[{u, name, type}]` | *anchor_u, priority* | 17 |
| `create_packed_layout(..., *, u, span, fields)` | PACKED layout `[{offset, name, type}]` | *priority* | 17 |
| `create_custom_layout(..., *, plugin, u_start, u_end)` | CUSTOM layout | *params, spacemap, priority* | 17 |
| `register_layout_plugin(name, plugin)` | attach the Python plugin | | 17 |
| `create_layout(layout_id, realm_id, name, kind)` | generic form | *u_start, u_end, priority, shape_id, params, spacemap, plugin, replace* | 17 |
| `get_layout(...)`, `layout_at(realm, u)`, `layout_candidates(realm, u)`, `route_address(realm, u, v)`, `layout_diagnostics(realm)` | routing | | 17 |
| `plan_layout_evolution(layout_id, ...)`, `evolve_layout(layout_id, ...)` | change a layout safely | *name, description, u_start, u_end, priority, add_fields, spacemap_update* | 17 |
| `logical_address(layout_id, object_id, field)`, `record_anchor(layout_id, object_id)` | object → cell address | | 10, 17 |

### 27.4 `NHyDB` – writing

| Method | Purpose | Key parameters | See |
|---|---|---|---|
| `put_record(layout_id, object_id, values)` | one record (dict) | | 3, 6 |
| `put(realm_id, u, values)` | one record (list) | *shape_id, extensions* | 6, 8 |
| `put_dense_batch(realm_id, u_start, rows)` | consecutive records | *shape_id, extensions_by_u* | 6, 8 |
| `put_object(layout_id, object_id, values)`, `put_objects_batch(layout_id, objects)` | records of any layout | *u_window* | 17 |
| `put_cell(realm_id, u, v, value)` | one cell | *type* | 7 |
| `put_points(realm_id, cells)` | many cells `(u, v, value[, type])` | *u_window* | 7 |
| `put_extensions(realm_id, u, mapping)`, `put_extension_list(realm_id, u, vs, values)`, `put_extension_range(realm_id, u, start_v, values)` | extension values | *merge=True* | 8 |
| `delete_record(layout_id, object_id)`, `delete(realm_id, u)` | delete records | | 6 |
| `delete_cell(realm_id, u, v)`, `delete_point(realm_id, u, v)` | delete cells | | 7 |

### 27.5 `NHyDB` – reading and addresses

| Method | Purpose | Key parameters | See |
|---|---|---|---|
| `get_record(layout_id, object_id)`, `get_object(...)` | one record | *include_addresses* (`get_object`) | 3, 6, 17 |
| `get(realm_id, u)` | dense record with extensions | *include_extensions, dyn_address, dyn_v* | 6, 12 |
| `get_value(realm_id, u, v)` | value of one cell | | 3, 7 |
| `get_cell(realm_id, u, v)` | cell with metadata | *dyn_address=True* | 7 |
| `get_internal(realm_id, u)` | raw values by `v` | | 6 |
| `get_record_view(layout_id, object_id)` | record with all cell numbers | *request_n* | 10 |
| `range(realm_id, u_from, u_to)`, `select(realm_id, u_from, u_to, columns)`, `select_us(realm_id, us, columns)` | lazy iterators | *dyn_address, dyn_v, include_extensions* | 12 |
| `iter_materialized_cells`, `iter_materialized_sparse_points`, `iter_materialized_record_ns` | everything stored | *realm_id* | 12 |
| `set_dyn_address_output(mode)` | default address indicators | *v* | 12 |
| `get_n(n)`, `get_n_cell(n)`, `get_n_record(n)`, `get_cell_n(n)`, `get_address(realm, u, v)` | read by N / address | *mode="auto"* | 10 |
| `describe_n(n)`, `resolve_n(n)` | explain an N | | 10 |
| `record_n_for_address(realm, u, v)`, `record_n_from_n(n)` | record anchor N | | 10 |
| `occupied_n(layout_id, object_id)`, `occupied_from_n(n)` | occupied cells of an object | *output="N"\|"ADDRESS"\|"BOTH"* | 10 |
| `cell_address(realm, u, v)`, `dyn_address(realm, u, v)` | address indicators | | 10 |

### 27.6 `NHyDB` – indexes, queries, NumPy, MATH

| Method / attribute | Purpose | Key parameters | See |
|---|---|---|---|
| `create_index(index_id, realm_id, name, field)` | VALUE (or other) index | *kind, options, tokenizer, casefold, ngram* | 13 |
| `create_token_index`, `create_trigram_index`, `create_fulltext_index` | text indexes | *tokenizer, casefold, ngram* | 13 |
| `build_indexes(index_ids=None)` | build / refresh | | 13 |
| `index_defs` | `{id: IndexDef}` | | 13 |
| `find_u(realm_id, filters)`, `find(realm_id, filters)` | index filter → `u` / records | *u_from, u_to* | 13 |
| `posting_count(index_id, value)`, `index_candidates(index_id, op, value, value2)`, `index_candidate_estimate(...)` | direct index access | | 13 |
| `fulltext_term_posting`, `fulltext_corpus`, `fulltext_payloads` | full-text internals | | 13 |
| `query(expr)`, `explain_query(expr)` | QueryEngine shortcuts | *realm, return_mode, select* | 14, 15 |
| `sql(text, params)`, `sql_numpy(text, params)`, `compile_sql(text, params)` | NHySQL | | 15, 16 |
| `put_numpy_batch(realm_id, u_start, columns)` | CLASSIC records from arrays | *shape_id* | 16 |
| `put_objects_numpy(layout_id, object_ids, columns)` | objects of any layout | *u_window* | 16 |
| `put_cells_numpy(realm_id, u, v, values)` | sparse cells | *type, u_window* | 16 |
| `read_numpy(realm_id, u_start, count, fields=None)` | columns + `_present` | | 16 |
| `create_math_layout(spec)`, `create_math_map_layout(name, parameters, *, u, V)` | MATH layouts | *layout_id, payload_type, side, realm_auto_limit* | 18 |
| `replace_math_spec(layout_id, spec)` | only before the first write | *payload_type* | 18 |
| `put_math(layout_id, inputs, payload)`, `put_math_batch(layout_id, rows)`, `put_math_numpy(layout_id, inputs, payloads)` | write MATH cells | *overwrite=False* | 18 |
| `get_math(layout_id, inputs)`, `get_math_cell(layout_id, inputs)` | read MATH cells | *dyn_address* | 18 |
| `describe_math(layout_id)`, `math_layouts` | MATH state | | 18 |

### 27.7 `nhydb.query`

| Name | Purpose | See |
|---|---|---|
| `QueryEngine(db)` | `execute`, `explain`, `search`, `execute_spec`, `execute_numpy`, `compile_sql`, `execute_sql`, `execute_sql_numpy`, `register_math_provider`, `math_providers`, `db` | 14, 15, 16 |
| `Q.field`, `Q.u`, `Q.v`, `Q.realm`, `Q.lane`, `Q.N`, `Q.record_N`, `Q.struct`, `Q.n`, `Q.math`, `Q.fulltext`, `Q.phrase`, `Q.all` | expression builder | 14 |
| `expr_from_spec(spec)`, `parse_sql(text)` | build expressions from data / parse SQL | 14, 15 |
| `QUERY_VERSION`, `SQL_LANGUAGE_VERSION` | version identifiers | 15 |

### 27.8 `nhydb.math`

| Name | Purpose | See |
|---|---|---|
| `MathExpression.compile(text, parameters)` / `.evaluate(**values)` | safe integer expressions | 18 |
| `MathMap.compile(name, parameters, *, u, V, side)` | mapping; `target`, `targets`, `to_dict`, `from_dict`, `fingerprint` | 18 |
| `MathMapLayoutSpec(name, mapping, realm_auto_limit)` | specification for `create_math_layout` | 18 |
| `dry_run(mapping, rows)` → `MathDryRunReport` | check a mapping without a database | 18 |
| `get_math_max_intermediate_bits()`, `set_math_max_intermediate_bits(bits)` | resource limit (process-wide) | 18 |
| `split_global_v`, `join_global_v`, `normalize_side`, `NL`, `NR`, `REALM_SPAN`, `DEFAULT_*` | helpers and constants | 18 |

### 27.9 Deliberately not covered by recipes

| Name | Reason |
|---|---|
| `nhydb.kernel`, `nhydb.index`, `nhydb.mathlayout` | stateless building blocks of the engine (codecs, tokenizers, BM25 kernel, realm planner); they exist for conformance tests and tooling. Use the database methods above instead. |
| `NHyDB.f` | read-only low-level file view (`seek`/`tell`) used by tests |
| expression and SQL classes in `nhydb.query` (`FieldPredicate`, `BoolExpr`, `SQLCompiledQuery`, `QueryContext`, `CandidateSet`, …) | data classes returned or consumed by the API; build expressions with `Q` instead |
| `MathMapTarget`, `MathDryRunReport`, `MathLayoutState` attributes | result objects; their fields are shown where they are returned (section 18) |
| `LayoutDef.contains_u`, `LayoutDef.to_dict`, `IndexDef.to_dict` | small helpers on result objects (sections 13, 17) |